# 面试问题：Double DQN 怎样降低 Q 值过估计，并完成可训练的 Agent 闭环？

## 可直接复述的回答主线

1. DQN 用在线 Q 网络同时选择和评估最大动作，估计噪声会通过 max 操作产生系统性正偏。
2. Double DQN 用 online network 选择下一动作，再让 target network 评估该动作，解耦选择与估值。
3. 训练 transition 必须包含 state、action、reward、next_state、terminated，终止样本不能继续 bootstrap。
4. 经验回放打破相邻样本相关性，target network 定期同步以稳定 Bellman 目标。
5. Q 值不是概率；行为阶段用 epsilon-greedy 探索，评估阶段直接 argmax。
6. 评测要展示环境轨迹、replay 样本、online/target 下一步 Q、TD target、梯度和逐起点 return。
7. 生产还需离线安全评估、reward 版本、行为策略校正、分布外动作门禁、回放治理和回滚。

后续实验会在同一批可读输入上依次展示基线、手写核心机制、训练过程、逐样本结果、失败修正与生产边界。

## 1. 真实案例与输入预览

案例是一个 7 个货位的仓库搬运通道。机器人从货位 0–5 中任一位置出发，动作是向左或向右；到达货位 6 完成拣货获得 +1，其余每步 -0.05，12 步未到达则截断。输出至少六条真实轨迹记录，并用六个起点的平均 return 和成功率比较策略。

In [1]:
import math  # 汇总强化学习梯度与回报指标。
import random  # 为环境探索和 replay 采样提供可复现随机源。
import warnings  # 过滤本地 PyTorch 环境的无关兼容警告。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 保持输出聚焦强化学习过程。
import torch  # 使用基础张量与自动微分手写 Q 网络和更新。
torch.manual_seed(411)  # 固定 Q 网络初始化与训练轨迹。
torch.set_num_threads(1)  # 固定 CPU 单线程提高可复现性。
class WarehouseChain:  # 定义七货位确定性仓库通道环境。
    def __init__(self, size=7, max_steps=12):  # 保存通道长度与截断步数。
        self.size = size  # 保存离散状态数量。
        self.max_steps = max_steps  # 保存单 episode 最大动作数。
        self.position = 0  # 初始化机器人位置。
        self.steps = 0  # 初始化当前 episode 步数。
    def observation(self):  # 把货位编号转为 one-hot 状态。
        return torch.nn.functional.one_hot(torch.tensor(self.position), num_classes=self.size).to(torch.float32)  # 返回七维状态向量。
    def reset(self, start=0):  # 从指定非终止货位开始新 episode。
        self.position = int(start)  # 写入起始货位。
        self.steps = 0  # 清零 episode 步数。
        return self.observation()  # 返回初始 one-hot 状态。
    def step(self, action):  # 执行左或右动作并返回环境 transition。
        previous = self.position  # 保存动作前货位供审计。
        self.position = max(0, self.position - 1) if int(action) == 0 else min(self.size - 1, self.position + 1)  # 按动作更新并裁剪边界。
        self.steps += 1  # 累加一次环境交互。
        terminated = self.position == self.size - 1  # 判断是否到达拣货终点。
        truncated = self.steps >= self.max_steps and not terminated  # 判断是否因时间上限截断。
        reward = 1.0 if terminated else -0.05  # 终点奖励一，其余动作支付时间成本。
        info = {"from": previous, "to": self.position, "action": "右" if action == 1 else "左", "reward": reward, "terminated": terminated, "truncated": truncated}  # 构造可读事件记录。
        return self.observation(), reward, terminated, truncated, info  # 返回下一状态与完整环境反馈。
preview_env = WarehouseChain()  # 创建用于输入预览的独立环境。
preview_state = preview_env.reset(start=1)  # 从货位一开始一段包含回退的轨迹。
preview_actions = [1, 0, 1, 1, 1, 1, 1]  # 定义七个可读动作事件。
preview_transitions = []  # 保存轨迹中的实际环境记录。
for action in preview_actions:  # 逐动作推进仓库通道。
    preview_state, reward, terminated, truncated, info = preview_env.step(action)  # 获取当前真实 transition。
    preview_transitions.append(info)  # 保存动作前后货位、奖励与终止标志。
    if terminated or truncated:  # 检查 episode 是否已经结束。
        break  # 到达终点或截断后停止预览。
print("教学实验输入：7货位仓库通道，state=one-hot(7)，actions={0:左,1:右}")  # 展示环境状态和动作 schema。
for index, transition in enumerate(preview_transitions):  # 逐条展示至少六条环境记录。
    print(f"transition-{index + 1}: {transition}")  # 输出真实 from/to/action/reward/terminal 字段。

教学实验输入：7货位仓库通道，state=one-hot(7)，actions={0:左,1:右}
transition-1: {'from': 1, 'to': 2, 'action': '右', 'reward': -0.05, 'terminated': False, 'truncated': False}
transition-2: {'from': 2, 'to': 1, 'action': '左', 'reward': -0.05, 'terminated': False, 'truncated': False}
transition-3: {'from': 1, 'to': 2, 'action': '右', 'reward': -0.05, 'terminated': False, 'truncated': False}
transition-4: {'from': 2, 'to': 3, 'action': '右', 'reward': -0.05, 'terminated': False, 'truncated': False}
transition-5: {'from': 3, 'to': 4, 'action': '右', 'reward': -0.05, 'terminated': False, 'truncated': False}
transition-6: {'from': 4, 'to': 5, 'action': '右', 'reward': -0.05, 'terminated': False, 'truncated': False}
transition-7: {'from': 5, 'to': 6, 'action': '右', 'reward': 1.0, 'terminated': True, 'truncated': False}


## 2. Baseline / 基线：始终向左

从货位 0–5 分别运行固定策略“始终向左”。它会停在边界并在 12 步后截断。与 Double DQN 使用完全相同的环境、起点和累计 return 指标。

In [2]:
def run_policy(policy, starts):  # 在多个起点执行同一策略并记录回报。
    rows = []  # 保存逐起点 episode 结果。
    for start in starts:  # 依次从六个非终止货位评估。
        environment = WarehouseChain()  # 为当前起点创建无残留状态的环境。
        state = environment.reset(start=start)  # 初始化当前 episode。
        total_reward = 0.0  # 初始化累计回报。
        path = [start]  # 保存机器人经过的货位路径。
        while True:  # 持续决策直到终止或截断。
            action = int(policy(state))  # 调用策略选择左或右。
            state, reward, terminated, truncated, info = environment.step(action)  # 执行动作并取得环境反馈。
            total_reward += reward  # 累加折扣前 episode reward。
            path.append(info["to"])  # 记录动作后的货位。
            if terminated or truncated:  # 检查 episode 是否结束。
                rows.append({"start": start, "return": total_reward, "success": terminated, "steps": environment.steps, "path": path})  # 保存当前完整 episode。
                break  # 结束当前起点评估。
    return rows  # 返回逐起点结果表。
baseline_rows = run_policy(lambda state: 0, list(range(6)))  # 评估始终向左的朴素策略。
baseline_mean_return = sum(row["return"] for row in baseline_rows) / len(baseline_rows)  # 计算六起点平均回报。
baseline_success_rate = sum(int(row["success"]) for row in baseline_rows) / len(baseline_rows)  # 计算六起点成功率。
print("Baseline：always-left policy")  # 标记下表为固定错误策略。
for row in baseline_rows:  # 逐起点展示实际走过的路径。
    print(f"start={row['start']} return={row['return']:.3f} success={row['success']} steps={row['steps']} path={row['path']}")  # 输出回报、终止和路径。
print(f"Baseline mean return={baseline_mean_return:.4f}，success rate={baseline_success_rate:.3f}")  # 展示后续 Agent 的同指标参照。

Baseline：always-left policy
start=0 return=-0.600 success=False steps=12 path=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
start=1 return=-0.600 success=False steps=12 path=[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
start=2 return=-0.600 success=False steps=12 path=[2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
start=3 return=-0.600 success=False steps=12 path=[3, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
start=4 return=-0.600 success=False steps=12 path=[4, 3, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
start=5 return=-0.600 success=False steps=12 path=[5, 4, 3, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0]
Baseline mean return=-0.6000，success rate=0.000


## 3. 底层实现：QNetwork、Replay Buffer 与 Double DQN Target

在线网络用 `argmax Q_online(s')` 选择动作，目标网络只评估这个动作。chosen action 的当前 Q 与无梯度 TD target 计算 Huber loss；target 每 20 次更新硬同步。

In [3]:
class QNetwork(torch.nn.Module):  # 定义从七维状态到两个动作 Q 值的 MLP。
    def __init__(self, state_dim=7, action_dim=2):  # 初始化两层隐藏网络。
        super().__init__()  # 注册 PyTorch 参数。
        self.network = torch.nn.Sequential(torch.nn.Linear(state_dim, 32), torch.nn.ReLU(), torch.nn.Linear(32, 32), torch.nn.ReLU(), torch.nn.Linear(32, action_dim))  # 映射 one-hot 状态到左右动作价值。
    def forward(self, states):  # 对一批状态输出未归一化 Q 值。
        return self.network(states)  # 返回批次乘动作数矩阵。
class ReplayBuffer:  # 定义固定容量的可复现经验回放。
    def __init__(self, capacity, seed):  # 初始化数据列表和独立随机源。
        self.capacity = capacity  # 保存最大 transition 数。
        self.data = []  # 创建 transition 存储列表。
        self.rng = random.Random(seed)  # 创建不污染探索随机流的采样器。
    def append(self, transition):  # 写入一条环境经验。
        if len(self.data) == self.capacity:  # 检查是否达到固定容量。
            self.data.pop(0)  # 移除最旧 transition。
        self.data.append(transition)  # 追加最新 transition。
    def sample(self, batch_size):  # 无放回采样并打包训练张量。
        rows = self.rng.sample(self.data, batch_size)  # 用独立随机源选择经验。
        states = torch.stack([row[0] for row in rows])  # 堆叠当前状态。
        actions = torch.tensor([row[1] for row in rows], dtype=torch.long)  # 构造动作编号。
        rewards = torch.tensor([row[2] for row in rows], dtype=torch.float32)  # 构造即时奖励。
        next_states = torch.stack([row[3] for row in rows])  # 堆叠下一状态。
        terminated = torch.tensor([row[4] for row in rows], dtype=torch.bool)  # 构造真正终止 mask。
        return states, actions, rewards, next_states, terminated  # 返回标准 DQN 批次。
class DoubleDQNAgent:  # 封装 online、target、replay 和一次更新逻辑。
    def __init__(self):  # 初始化两个同构网络与优化器。
        self.online = QNetwork()  # 创建参与梯度更新的在线网络。
        self.target = QNetwork()  # 创建稳定 Bellman 估值网络。
        self.target.load_state_dict(self.online.state_dict())  # 在训练前同步两套参数。
        self.optimizer = torch.optim.Adam(self.online.parameters(), lr=0.01)  # 仅优化 online network。
        self.replay = ReplayBuffer(800, seed=412)  # 创建独立采样经验回放。
        self.exploration_rng = random.Random(413)  # 创建 epsilon-greedy 独立随机源。
        self.update_count = 0  # 初始化参数更新次数。
    def select_action(self, state, epsilon):  # 使用 epsilon-greedy 选择训练动作。
        if self.exploration_rng.random() < epsilon:  # 按 epsilon 判断是否探索。
            return self.exploration_rng.randrange(2)  # 随机选择左或右。
        with torch.no_grad():  # 在行为选择时关闭梯度。
            return int(self.online(state.unsqueeze(0)).argmax(dim=1).item())  # 选择在线网络最大 Q 动作。
    def train_step(self, batch_size=32, gamma=0.95):  # 从 replay 执行一次 Double DQN 更新。
        states, actions, rewards, next_states, terminated = self.replay.sample(batch_size)  # 采样独立 transition 批次。
        current_all_q = self.online(states)  # 计算当前状态两个动作 Q 值。
        chosen_q = current_all_q.gather(1, actions[:, None]).squeeze(1)  # 只读取实际执行动作的 Q。
        with torch.no_grad():  # 构造不参与反向传播的 Bellman target。
            online_next_q = self.online(next_states)  # 让 online 网络负责下一动作选择。
            next_actions = online_next_q.argmax(dim=1)  # 取得 Double DQN 选择动作。
            target_next_q = self.target(next_states)  # 让 target 网络负责动作估值。
            selected_next_q = target_next_q.gather(1, next_actions[:, None]).squeeze(1)  # 读取已选择动作的 target Q。
            td_targets = rewards + gamma * (~terminated).to(torch.float32) * selected_next_q  # 对非终止样本执行 bootstrap。
        loss = torch.nn.functional.smooth_l1_loss(chosen_q, td_targets)  # 用 Huber loss 降低异常 TD error 影响。
        self.optimizer.zero_grad(set_to_none=True)  # 清除上一步 online 梯度。
        loss.backward()  # 对在线 Q 网络执行真实反向传播。
        gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in self.online.parameters() if parameter.grad is not None))  # 汇总在线网络梯度二范数。
        self.optimizer.step()  # 应用 Adam 更新在线参数。
        self.update_count += 1  # 累加一次参数更新。
        if self.update_count % 20 == 0:  # 每二十次更新检查同步条件。
            self.target.load_state_dict(self.online.state_dict())  # 硬同步 target network 参数。
        debug = {"states": states, "actions": actions, "rewards": rewards, "terminated": terminated, "online_next_q": online_next_q, "next_actions": next_actions, "target_next_q": target_next_q, "selected_next_q": selected_next_q, "td_targets": td_targets, "chosen_q": chosen_q.detach()}  # 保存 Double DQN 目标构造证据。
        return loss.item(), gradient_norm, debug  # 返回损失、梯度与当前采样中间量。
agent = DoubleDQNAgent()  # 创建待训练仓库 Agent。
training_history = []  # 保存 episode return、epsilon 和 TD 更新轨迹。
latest_debug = None  # 预留最近一次 replay 更新中间量。
for episode in range(180):  # 运行完整环境交互与经验回放训练。
    environment = WarehouseChain()  # 为当前 episode 创建干净环境。
    state = environment.reset(start=episode % 6)  # 轮流覆盖六个起始货位。
    epsilon = max(0.05, 0.9 - episode / 160.0)  # 从高探索线性退火到百分之五。
    episode_return = 0.0  # 初始化当前累计回报。
    episode_losses = []  # 保存当前 episode 内 TD loss。
    while True:  # 持续与环境交互直到结束。
        action = agent.select_action(state, epsilon)  # 根据 online Q 与探索率选择动作。
        next_state, reward, terminated, truncated, info = environment.step(action)  # 执行动作得到真实 transition。
        agent.replay.append((state.clone(), action, reward, next_state.clone(), terminated))  # 把真正 terminated 与状态写入 replay。
        state = next_state  # 推进到下一状态。
        episode_return += reward  # 累加当前 episode 回报。
        if len(agent.replay.data) >= 32:  # replay 足够大后开始随机批训练。
            loss_value, gradient_norm, latest_debug = agent.train_step()  # 执行一次 Double DQN 参数更新。
            episode_losses.append(loss_value)  # 保存当前 TD loss。
        if terminated or truncated:  # 检查环境是否结束。
            break  # 离开当前 episode 交互循环。
    if episode % 45 == 0 or episode == 179:  # 每四十五个 episode 保存训练摘要。
        mean_loss = sum(episode_losses) / len(episode_losses) if episode_losses else float("nan")  # 计算当前 episode 平均更新损失。
        training_history.append({"episode": episode, "return": episode_return, "epsilon": epsilon, "mean_loss": mean_loss, "replay_size": len(agent.replay.data), "updates": agent.update_count})  # 保存策略与优化轨迹。
print("Double DQN训练轨迹=", training_history)  # 展示探索率、回报、loss 和 replay 增长。
print("最近replay batch前6条 action/reward/terminated=", list(zip(latest_debug["actions"][:6].tolist(), latest_debug["rewards"][:6].tolist(), latest_debug["terminated"][:6].tolist())))  # 展示真实采样 transition 字段。
print("最近online next Q前3行=", torch.round(latest_debug["online_next_q"][:3] * 1000) / 1000)  # 展示动作选择依据。
print("最近next_action/target_selected/TD_target前6条=", list(zip(latest_debug["next_actions"][:6].tolist(), latest_debug["selected_next_q"][:6].tolist(), latest_debug["td_targets"][:6].tolist())))  # 展示解耦选择与估值后的 Bellman 目标。

Double DQN训练轨迹= [{'episode': 0, 'return': 0.55, 'epsilon': 0.9, 'mean_loss': nan, 'replay_size': 10, 'updates': 0}, {'episode': 45, 'return': -0.6, 'epsilon': 0.61875, 'mean_loss': 4.347824718327805e-05, 'replay_size': 418, 'updates': 387}, {'episode': 90, 'return': 0.65, 'epsilon': 0.3375, 'mean_loss': 1.3674928140972042e-06, 'replay_size': 686, 'updates': 655}, {'episode': 135, 'return': 0.9, 'epsilon': 0.05625000000000002, 'mean_loss': 3.232072328292664e-07, 'replay_size': 800, 'updates': 862}, {'episode': 179, 'return': 1.0, 'epsilon': 0.05, 'mean_loss': 2.2853001269140805e-08, 'replay_size': 800, 'updates': 1020}]
最近replay batch前6条 action/reward/terminated= [(1, 1.0, True), (1, -0.05000000074505806, False), (1, -0.05000000074505806, False), (0, -0.05000000074505806, False), (1, -0.05000000074505806, False), (0, -0.05000000074505806, False)]
最近online next Q前3行= tensor([[0.6710, 0.8300],
        [0.8050, 1.0000],
        [0.7150, 0.9000]])
最近next_action/target_selected/TD_target前6条=

## 4. 六起点策略结果与结果解读

关闭探索，用在线网络 argmax 从六个起点各运行一次，逐条输出 Q 值、路径、return 和成功标志，并与始终向左基线使用同一指标比较。

In [4]:
def greedy_agent_policy(state):  # 定义评估阶段无探索的在线 Q 策略。
    with torch.no_grad():  # 关闭评估梯度。
        return int(agent.online(state.unsqueeze(0)).argmax(dim=1).item())  # 返回最大 Q 动作。
agent_rows = run_policy(greedy_agent_policy, list(range(6)))  # 在同一六起点环境评估训练后 Agent。
agent_mean_return = sum(row["return"] for row in agent_rows) / len(agent_rows)  # 计算训练后六起点平均回报。
agent_success_rate = sum(int(row["success"]) for row in agent_rows) / len(agent_rows)  # 计算训练后成功率。
with torch.no_grad():  # 读取所有离散状态的最终 Q 表。
    state_q_table = agent.online(torch.eye(7))  # 对七个 one-hot 状态计算左右动作价值。
print("start  Q(left)  Q(right)  action  return  success  path")  # 输出逐起点策略结果表头。
for row in agent_rows:  # 逐起点展示策略与路径。
    q_values = state_q_table[row["start"]]  # 读取当前起点左右 Q 值。
    action_name = "右" if int(q_values.argmax()) == 1 else "左"  # 把最大 Q 动作转换为中文。
    print(f"{row['start']:>5} {q_values[0].item():>8.3f} {q_values[1].item():>9.3f} {action_name:>7} {row['return']:>7.3f} {str(row['success']):>8} {row['path']}")  # 输出 Q、动作、回报和完整路径。
print(f"结果解读：always-left mean return={baseline_mean_return:.4f}/success={baseline_success_rate:.3f}；Double DQN={agent_mean_return:.4f}/success={agent_success_rate:.3f}。")  # 解释同环境策略收益。

start  Q(left)  Q(right)  action  return  success  path
    0    0.470     0.548       右   0.750     True [0, 1, 2, 3, 4, 5, 6]
    1    0.470     0.628       右   0.800     True [1, 2, 3, 4, 5, 6]
    2    0.548     0.715       右   0.850     True [2, 3, 4, 5, 6]
    3    0.629     0.805       右   0.900     True [3, 4, 5, 6]
    4    0.715     0.900       右   0.950     True [4, 5, 6]
    5    0.805     1.000       右   1.000     True [5, 6]
结果解读：always-left mean return=-0.6000/success=0.000；Double DQN=0.8750/success=1.000。


## 5. 失败案例与修正：同一个 max 同时选择并估值

构造一个带估计噪声的下一状态：online 更偏好动作 0，但 target 对动作 1 有偶然高估。普通 DQN 直接取 target max=6；Double DQN 坚持评估 online 选出的动作 0，得到 1，避免把另一个网络的高噪声动作当成目标。

In [5]:
synthetic_online_q = torch.tensor([[5.0, 4.0]])  # 构造在线网络认为动作零更优的 Q。
synthetic_target_q = torch.tensor([[1.0, 6.0]])  # 构造目标网络对另一动作偶然高估的 Q。
synthetic_reward = 0.2  # 定义当前非终止即时奖励。
synthetic_gamma = 0.9  # 定义折扣因子。
plain_selected_value = float(synthetic_target_q.max(dim=1).values.item())  # 普通 DQN 直接取 target 网络最大值。
double_selected_action = int(synthetic_online_q.argmax(dim=1).item())  # Double DQN 先由 online 选择动作。
double_selected_value = float(synthetic_target_q[0, double_selected_action].item())  # 再由 target 评估同一动作。
plain_target = synthetic_reward + synthetic_gamma * plain_selected_value  # 计算耦合 max 的过高目标。
double_target = synthetic_reward + synthetic_gamma * double_selected_value  # 计算解耦选择与估值的目标。
print(f"错误行为：plain DQN使用max Q_target={plain_selected_value:.1f}，TD target={plain_target:.2f}")  # 展示目标网络噪声被 max 放大。
print(f"修正行为：online选择action={double_selected_action}，target只估值该动作={double_selected_value:.1f}，TD target={double_target:.2f}")  # 展示 Double DQN 职责分离。

错误行为：plain DQN使用max Q_target=6.0，TD target=5.60
修正行为：online选择action=0，target只估值该动作=1.0，TD target=1.10


## 6. 生产边界

七状态确定性环境不代表真实仓储。生产强化学习需要离线日志与行为策略概率、reward 和环境版本、counterfactual/OPE、安全动作白名单、分布外检测、延迟与故障降级、replay 隐私治理、shadow/canary 发布，以及按场景监控 return、TD error、Q 值漂移和探索风险。

In [6]:
double_dqn_diagnostics = {"states": 7, "actions": 2, "episodes": 180, "replay_size": len(agent.replay.data), "updates": agent.update_count, "baseline_mean_return": baseline_mean_return, "agent_mean_return": agent_mean_return, "baseline_success_rate": baseline_success_rate, "agent_success_rate": agent_success_rate, "plain_synthetic_target": plain_target, "double_synthetic_target": double_target}  # 汇总环境、训练、策略和过估计指标。
print("生产监控快照：", double_dqn_diagnostics)  # 输出 Double DQN 系统应持续观察的信号。

生产监控快照： {'states': 7, 'actions': 2, 'episodes': 180, 'replay_size': 800, 'updates': 1020, 'baseline_mean_return': -0.6, 'agent_mean_return': 0.875, 'baseline_success_rate': 0.0, 'agent_success_rate': 1.0, 'plain_synthetic_target': 5.6000000000000005, 'double_synthetic_target': 1.1}


## 7. 最小回归测试

最后一格只保护环境记录、真实更新、策略收益、终止 mask 和 Double DQN 解耦目标。

In [7]:
assert len(preview_transitions) >= 6 and len(agent.replay.data) >= 32  # 保证展示了足够真实交互并填充 replay。
assert agent.update_count > 0 and all(row["updates"] >= 0 for row in training_history)  # 保证完整训练循环实际执行参数更新。
assert all(parameter.grad is not None and torch.isfinite(parameter.grad).all() for parameter in agent.online.parameters())  # 保证在线 Q 网络获得有限梯度。
assert agent_mean_return > baseline_mean_return and agent_success_rate >= 0.80  # 保证同一六起点环境策略明显优于固定基线。
assert torch.all(latest_debug["td_targets"][latest_debug["terminated"]] == latest_debug["rewards"][latest_debug["terminated"]])  # 保证终止 transition 不再 bootstrap。
assert plain_target > double_target and double_selected_action == 0  # 保证耦合 max 过估计可复现并被 Double DQN 修正。